# Is inter-brain synchrony (IBS) associated with subjective involvement?

## Load useful libs

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from hypyp import analyses

## Get Raw data

In [3]:
raw = mne.io.read_raw_brainvision(
    "data/ds007471/sub-01/eeg/sub-01_task-jointaction_eeg.vhdr",
    preload=True,
)

print(raw.info)
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f}")

Extracting parameters from data/ds007471/sub-01/eeg/sub-01_task-jointaction_eeg.vhdr...
Setting channel info structure...


/var/folders/2x/v_ydgf450nd23wnb3_hcjtw40000gn/T/ipykernel_28672/690757239.py:1: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(


Reading 0 ... 2318619  =      0.000 ...  2318.619 secs...
<Info | 7 non-empty values
 bads: []
 ch_names: VEOG_b_R, VEOG_a_R, F7_R, F3_R, Fz_R, F4_R, F8_R, FC5_R, FC1_R, ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 1.0 Hz
 lowpass: 125.0 Hz
 meas_date: 2022-11-30 14:50:15 UTC
 nchan: 64
 projs: []
 sfreq: 1000.0 Hz
>
Sampling rate: 1000.0 Hz
Duration: 2318.6


**Raw Data:**
- Number of Channels: 64 Channels simultaneously (32 per person)
- Two eye-movement-channels: `VEOG_b_R` and `VEOG_a_R` 
- Sampling rate: 1000 Hz
- Duration: 2,3186 seconds ≈ 38.6 minutes
- Frequency Range 1-125 Hz

## Split into two participants and drop non-brain channels

In [15]:
# VEOG_b, VEOG_a, HEOG_l, HEOG_r are eye-movement electrodes, not brain signal, and ref_lm is the linked-mastoid reference
non_eeg = {"VEOG_b", "VEOG_a", "HEOG_l", "HEOG_r", "ref_lm"}

eeg_R = [ch for ch in raw.ch_names if ch.endswith("_R") and ch[:-2] not in non_eeg]
eeg_L = [ch for ch in raw.ch_names if ch.endswith("_L") and ch[:-2] not in non_eeg]

raw_R = raw.copy().pick(eeg_R)
raw_L = raw.copy().pick(eeg_L)

raw_R.rename_channels({name: name.removesuffix("_R") for name in raw_R.ch_names})
raw_L.rename_channels({name: name.removesuffix("_L") for name in raw_L.ch_names})

print(raw_L)

<RawBrainVision | sub-01_task-jointaction_eeg.eeg, 27 x 2318620 (2318.6 s), ~477.6 MiB, data loaded>


## Set Montage

In [20]:
raw_R.set_montage("standard_1020", on_missing="warn")
raw_L.set_montage("standard_1020", on_missing="warn")

# Give MNE Information on event-markers
events, event_id = mne.events_from_annotations(raw)

# Look up the numeric code for marker Sn, e.g. event_code(105) for S105
def event_code(n):
    return event_id[f"Stimulus/S{n:>3}"]

print(f"{len(events)} total markers")
print(f"Event types: {event_id}")

Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S 10'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S105'), np.str_('Stimulus/S106'), np.str_('Stimulus/S107'), np.str_('Stimulus/S128'), np.str_('Stimulus/S130'), np.str_('Stimulus/S131'), np.str_('Stimulus/S132')]
2229 total markers
Event types: {np.str_('Stimulus/S  1'): 1, np.str_('Stimulus/S  2'): 2, np.str_('Stimulus/S  3'): 3, np.str_('Stimulus/S  4'): 4, np.str_('Stimulus/S 10'): 10, np.str_('Stimulus/S 11'): 11, np.str_('Stimulus/S105'): 105, np.str_('Stimulus/S106'): 106, np.str_('Stimulus/S107'): 107, np.str_('Stimulus/S128'): 128, np.str_('Stimulus/S130'): 130, np.str_('Stimulus/S131'): 131, np.str_('Stimulus/S132'): 132}


## Filter out practice trials and label each real trial's condition

In [21]:
s1_sample = events[events[:, 2] == event_code(1)][0, 0]

all_trial_events = events[events[:, 2] == event_code(105)]
real_trial_events = all_trial_events[all_trial_events[:, 0] > s1_sample]

print(f"{len(all_trial_events)} total S105 markers "
      f"({len(all_trial_events) - len(real_trial_events)} practice, {len(real_trial_events)} real test trials)")

s10, s11 = event_code(10), event_code(11)
condition_events = events[np.isin(events[:, 2], [s10, s11])]

trial_condition = []
for sample in real_trial_events[:, 0]:
    preceding = condition_events[condition_events[:, 0] < sample]
    trial_condition.append("duet" if preceding[-1, 2] == s10 else "constant_pitch")
trial_condition = np.array(trial_condition)

print(f"Duet trials: {(trial_condition == 'duet').sum()}, "
      f"constant-pitch trials: {(trial_condition == 'constant_pitch').sum()}")

63 total S105 markers (6 practice, 57 real test trials)
Duet trials: 28, constant-pitch trials: 29
